# Step 01_02_03 -- Raw Schema DESCRIBE: aoestats

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoestats
**Question:** What are the column names and types for every `*_raw` table in
the aoestats DuckDB? This establishes a baseline schema snapshot used by
downstream feature engineering steps.
**Invariants applied:** #6 (reproducibility — SQL inlined in artifact), #9 (step scope: query)
**Step scope:** query
**Type:** Read-only — no DuckDB writes, no new tables, no schema changes

In [1]:
import json

import duckdb

from rts_predict.common.notebook_utils import get_reports_dir, setup_notebook_logging
from rts_predict.games.aoe2.config import AOESTATS_DB_FILE

logger = setup_notebook_logging()
logger.info("DB_FILE: %s", AOESTATS_DB_FILE)

12:58:17 INFO notebook: DB_FILE: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/data/db/db.duckdb


## 1. Connect to DuckDB (read-only)

In [2]:
con = duckdb.connect(str(AOESTATS_DB_FILE), read_only=True)
print(f"Connected (read-only): {AOESTATS_DB_FILE}")

Connected (read-only): /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/data/db/db.duckdb


## 2. DESCRIBE all *_raw tables

Tables covered:
- `matches_raw`
- `players_raw`
- `overviews_raw`

In [3]:
RAW_TABLES = [
    "matches_raw",
    "players_raw",
    "overviews_raw",
]

schemas: dict[str, list[dict]] = {}

for table in RAW_TABLES:
    df = con.execute(f"DESCRIBE {table}").df()
    schemas[table] = df.to_dict(orient="records")
    print(f"\n=== DESCRIBE {table} ({len(df)} columns) ===")
    print(df.to_string(index=False))


=== DESCRIBE matches_raw (18 columns) ===
      column_name              column_type null  key default extra
              map                  VARCHAR  YES None    None  None
started_timestamp TIMESTAMP WITH TIME ZONE  YES None    None  None
         duration                   BIGINT  YES None    None  None
     irl_duration                   BIGINT  YES None    None  None
          game_id                  VARCHAR  YES None    None  None
          avg_elo                   DOUBLE  YES None    None  None
      num_players                   BIGINT  YES None    None  None
       team_0_elo                   DOUBLE  YES None    None  None
       team_1_elo                   DOUBLE  YES None    None  None
  replay_enhanced                  BOOLEAN  YES None    None  None
      leaderboard                  VARCHAR  YES None    None  None
           mirror                  BOOLEAN  YES None    None  None
            patch                   BIGINT  YES None    None  None
   raw_match_type  

## 3. Write artifact

In [4]:
artifacts_dir = (
    get_reports_dir("aoe2", "aoestats")
    / "artifacts" / "01_exploration" / "02_eda"
)
artifacts_dir.mkdir(parents=True, exist_ok=True)

artifact_data = {
    "step": "01_02_03",
    "dataset": "aoestats",
    "schemas": schemas,
}

artifact_path = artifacts_dir / "01_02_03_raw_schema_describe.json"
artifact_path.write_text(json.dumps(artifact_data, indent=2, default=str))
logger.info("Artifact written: %s", artifact_path)

print(f"\nArtifact written: {artifact_path}")
for table, cols in schemas.items():
    print(f"  {table}: {len(cols)} columns")

12:58:17 INFO notebook: Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/02_eda/01_02_03_raw_schema_describe.json



Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/02_eda/01_02_03_raw_schema_describe.json
  matches_raw: 18 columns
  players_raw: 14 columns
  overviews_raw: 9 columns


In [5]:
con.close()